# Set PYSPARK_SUBMIT_ARGS to match your working batch file launcher

In [1]:
import os
import sys
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    '--conf spark.driver.extraClassPath="C:/data/spark/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar" '
    "pyspark-shell"
)

# 2. Ensure Python paths align for the worker processes
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Connection

## initilise connection

In [2]:
from pyspark.sql import SparkSession

# --- Define Paths ---
# Adjust the local path to match your OS requirements (e.g., C:/data/... for Windows)
RPT_WAREHOUSE_PATH_LOCAL = "/data/data_files/iceberg/WideWorldImportersDW"
RPT_WAREHOUSE_PATH_MINIO = "s3a://iceberg/iceberg/WideWorldImportersDW"

# --- 1. Initialize Combined SparkSession ---
spark = SparkSession.builder \
    .appName("Iceberg Local to MinIO Transfer") \
    .config("spark.jars.packages", 
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    \
    .config("spark.sql.catalog.local_rpt", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local_rpt.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config("spark.sql.catalog.local_rpt.warehouse", f"file:///{RPT_WAREHOUSE_PATH_LOCAL.lstrip('/')}") \
    \
    .config("spark.sql.catalog.minio_rpt", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.minio_rpt.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config("spark.sql.catalog.minio_rpt.warehouse", RPT_WAREHOUSE_PATH_MINIO) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

## Create Namespaces in MinIO

In [3]:



# MinIO needs the namespaces to exist before creating tables inside them
spark.sql("CREATE NAMESPACE IF NOT EXISTS minio_rpt.dimension")
spark.sql("CREATE NAMESPACE IF NOT EXISTS minio_rpt.fact")


DataFrame[]

## Define Tables to Transfer

In [4]:
tables_to_copy = {
    "dimension": [
        "payment_method", "supplier", "city", 
        "stock_item", "customer", "transaction_type", "employee"
    ],
    "fact": [
        "purchase", "stock_holding", "order", 
        "movement", "sale", "transaction"
    ]
}

##  Execute the Transfer

In [5]:

for namespace, tables in tables_to_copy.items():
    for table_name in tables:
        source_identifier = f"local_rpt.{namespace}.{table_name}"
        dest_identifier = f"minio_rpt.{namespace}.{table_name}"
        
        print(f"Reading from: {source_identifier}")
        df = spark.table(source_identifier)
        
        print(f"Writing to:   {dest_identifier}")
        # .createOrReplace() will create the table if it doesn't exist, 
        # or overwrite it if you are running this multiple times
        df.writeTo(dest_identifier).createOrReplace()
        
        print(f"Successfully transferred {table_name}.\n")

print("All tables successfully transferred to MinIO!")

Reading from: local_rpt.dimension.payment_method
Writing to:   minio_rpt.dimension.payment_method
Successfully transferred payment_method.

Reading from: local_rpt.dimension.supplier
Writing to:   minio_rpt.dimension.supplier
Successfully transferred supplier.

Reading from: local_rpt.dimension.city
Writing to:   minio_rpt.dimension.city
Successfully transferred city.

Reading from: local_rpt.dimension.stock_item
Writing to:   minio_rpt.dimension.stock_item
Successfully transferred stock_item.

Reading from: local_rpt.dimension.customer
Writing to:   minio_rpt.dimension.customer
Successfully transferred customer.

Reading from: local_rpt.dimension.transaction_type
Writing to:   minio_rpt.dimension.transaction_type
Successfully transferred transaction_type.

Reading from: local_rpt.dimension.employee
Writing to:   minio_rpt.dimension.employee
Successfully transferred employee.

Reading from: local_rpt.fact.purchase
Writing to:   minio_rpt.fact.purchase
Successfully transferred purchase.